# 1. Fourier fitting and explicit validation

Only this notebook generates Fourier coefficients. It saves immutable fits; no classical dynamics are triggered until you explicitly populate `selected_for_dynamics`. Raw fit screening and accepted reduced fits are shown **separately**, so rejected candidates do not introduce misleading NaNs into the saved-fit table.

In [8]:
from pathlib import Path
import sys
import pandas as pd

# Locate the checkout so notebooks work without an editable install.
# Supported working directories: repository root or a directory inside it.
_here = Path.cwd().resolve()
ROOT = next(
    (directory for directory in (_here, *_here.parents)
     if (directory / "src" / "bosonic_algo" / "__init__.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the bosonic_algo checkout from the notebook kernel's "
        "working directory. Start Jupyter in the repository root or its "
        "notebooks/ folder (or change the kernel working directory). "
        "For another environment, install from the repository root with "
        "'python -m pip install -e .'."
    )
_src = str(ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from bosonic_algo import generate_fits, read_yaml, selected_fit_classical_validation, list_fits
from bosonic_algo import fourier as _fourier
if getattr(_fourier, "SUMMARY_SCHEMA_VERSION", 0) < 2:
    raise RuntimeError(
        "Outdated Fourier workflow detected in the Jupyter kernel. "
        "Run 'git pull' on the refactor branch and restart the notebook kernel. "
        "The old version mixes rejected candidates with saved fits, causing NaNs."
    )
print("Using Fourier fitting module:", Path(_fourier.__file__).resolve())
FIT_ROOT = ROOT / "data" / "fits"
RESULT_ROOT = ROOT / "results"
settings = read_yaml(ROOT / "configs" / "fourier.yaml")

Using Fourier fitting module: /home/iren/quantum_algorithm/Shin-Metiu/bosonic_algo/src/bosonic_algo/fourier.py


## Generate reduced fits (no time evolution)

`rows` contains **only saved reduced Fourier fits**, with uniform columns. All raw candidates, including rejected ones, are written to `data/fits/raw_fit_summary.csv`.

In [4]:
rows = generate_fits(settings, root=FIT_ROOT)
saved_df = pd.DataFrame(rows)
print(f"Saved reduced fits: {len(rows)}")
if saved_df.empty:
    print("No raw candidates passed screening; inspect the raw summary below.")
else:
    threshold = settings.get("raw_v_rmse_max", 0.01)
    above = saved_df[saved_df["v_rmse"] > threshold]
    if len(above):
        print(f"Note: {len(above)} reduced fits exceed raw-screen threshold "
              f"({threshold:g}) AFTER selecting sparse modes. Review their v_rmse and forces.")
    if saved_df.isna().any().any():
        raise RuntimeError("Saved-fit table contains missing values; verify imported module version.")
saved_df

Saved reduced fits: 8
Note: 2 reduced fits exceed raw-screen threshold (0.01) AFTER selecting sparse modes. Review their v_rmse and forces.


,fit_id,status,basis,fourier_order_R,fourier_order_r,fourier_box_R,fourier_box_r,ridge_lambda,order_strength,order_power,...,force_R_rmse,force_r_rmse,force_combined_rmse,n_modes,max_amplitude,sum_amplitude,max_force_scale,max_theta,n_tgifs_terms,path
0,fit_eda7b7b165d14318,saved,half_lattice,14,14,20.0,60.0,0.0001,10.0,1.0,...,0.036799,0.005149,0.026274,50,0.121123,0.615198,0.022249,0.602952,99,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
1,fit_d250fb158c7e93db,saved,half_lattice,14,14,20.0,80.0,0.0001,10.0,1.0,...,0.025601,0.001652,0.018140,50,0.364943,1.300083,0.052106,1.824345,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
2,fit_9600c945824059c7,saved,half_lattice,14,14,40.0,60.0,0.0001,10.0,1.0,...,0.003503,0.006508,0.005226,50,0.083229,0.662750,0.015901,0.414271,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
3,fit_342a4798415698a4,saved,half_lattice,14,14,40.0,80.0,0.0001,10.0,1.0,...,0.003101,0.001570,0.002458,50,0.209882,1.592598,0.046145,1.047972,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
4,fit_701eb8a6de657f7c,saved,half_lattice,14,14,60.0,60.0,0.0001,10.0,1.0,...,0.001569,0.004846,0.003602,50,0.065850,0.784369,0.019017,0.327751,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
5,fit_f33d5a618d552ac9,saved,half_lattice,14,14,60.0,80.0,0.0001,10.0,1.0,...,0.001160,0.001791,0.001509,50,0.382642,2.966003,0.059845,1.912593,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
6,fit_7277d62b90160302,saved,half_lattice,14,14,80.0,60.0,0.0001,10.0,1.0,...,0.001309,0.004846,0.003550,50,0.098330,1.032846,0.026946,0.489472,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
7,fit_058cc3aeeede3a81,saved,half_lattice,14,14,80.0,80.0,0.0001,10.0,1.0,...,0.001549,0.002572,0.002123,50,0.573153,4.327645,0.145263,2.853910,100,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...


## Raw Fourier-fit screening

The raw table contains **every** Fourier candidate and its raw potential/force metrics. A `rejected` status means the raw potential RMSE exceeds `raw_v_rmse_max`; it was not reduced or propagated. This is expected, not missing numerical data.

In [5]:
raw_df = pd.read_csv(FIT_ROOT / "raw_fit_summary.csv")
print("Raw candidates:", len(raw_df), "| Passed:", (raw_df["status"] == "passed").sum(),
      "| Rejected:", (raw_df["status"] == "rejected").sum())
print("Counts by box_r and status:")
print(raw_df.groupby(["fourier_box_r", "status"]).size().unstack(fill_value=0))
if raw_df.select_dtypes(include="number").isna().any().any():
    raise RuntimeError("Non-finite numeric metrics in raw-fit summary; inspect the solver.")
raw_df

Raw candidates: 16 | Passed: 8 | Rejected: 8
Counts by box_r and status:
status         passed  rejected
fourier_box_r                  
20.0                0         4
40.0                0         4
60.0                4         0
80.0                4         0


,basis,fourier_order_R,fourier_order_r,fourier_box_R,fourier_box_r,ridge_lambda,order_strength,order_power,raw_fit_checksum,status,...,v_max_abs_error,force_R_rmse,force_r_rmse,force_combined_rmse,n_modes,max_amplitude,sum_amplitude,max_force_scale,max_theta,n_tgifs_terms
0,half_lattice,14,14,20.0,20.0,0.0001,10.0,1.0,1c8e68d48fd7ddbbc78b078f7f4a1e07a968a52b368d11...,rejected,...,0.257975,0.030669,0.017406,0.024936,364,0.032125,0.233761,0.018923,0.160313,727
1,half_lattice,14,14,20.0,40.0,0.0001,10.0,1.0,5556ed3622ea765b96640e9427d1fe0cfce77dd3ad2997...,rejected,...,0.252828,0.040979,0.039796,0.040392,364,0.094574,0.811243,0.020872,0.461707,727
2,half_lattice,14,14,20.0,60.0,0.0001,10.0,1.0,fdadb7588cdbb844291db132915f89715cd6839ea2599c...,passed,...,0.004703,0.038670,0.000881,0.027351,364,0.120797,0.740431,0.022411,0.601346,727
3,half_lattice,14,14,20.0,80.0,0.0001,10.0,1.0,71ee2d9da000592ce91fc90a02156c355f8025137908a8...,passed,...,0.000448,0.034519,0.000316,0.024410,364,0.114525,1.311328,0.020320,0.572103,727
4,half_lattice,14,14,40.0,20.0,0.0001,10.0,1.0,1b323fe763334f61102a97ea3376426bc2c6deb0ea930b...,rejected,...,0.257975,0.008325,0.017406,0.013643,364,0.033701,0.274205,0.014787,0.162881,728
5,half_lattice,14,14,40.0,40.0,0.0001,10.0,1.0,6f99046a3c34ba72c9a8a5e06d87ffd8bf78d173da5d3f...,rejected,...,0.252828,0.005031,0.039796,0.028364,364,0.056574,0.790416,0.015231,0.275966,728
6,half_lattice,14,14,40.0,60.0,0.0001,10.0,1.0,19ed1ca043f368352608c2e6d528cbb50bc235e2e60c2e...,passed,...,0.004703,0.003548,0.000881,0.002585,364,0.073370,0.724735,0.014787,0.365234,728
7,half_lattice,14,14,40.0,80.0,0.0001,10.0,1.0,64592c3ebbf1d1cde330f78ce742a39d97dec4daab09fa...,passed,...,0.000442,0.003312,0.000343,0.002354,364,0.064423,1.336585,0.012917,0.321393,728
8,half_lattice,14,14,60.0,20.0,0.0001,10.0,1.0,919b3f6cda033342306dd2cbf7ba219b35c5496c9717d3...,rejected,...,0.257977,0.007721,0.017406,0.013464,364,0.060531,0.407399,0.019017,0.302515,728
9,half_lattice,14,14,60.0,40.0,0.0001,10.0,1.0,fcfb2e2c46764b4a1552b8be9b6f6fb11b6e801c861e0f...,rejected,...,0.252831,0.003536,0.039796,0.028251,364,0.061849,0.890813,0.019430,0.309137,728


## Choose fit IDs from this sweep

The table below contains only saved fits from the current sweep, **not historical metadata**. The old `list_fits()` directory scan can mix runs created with different reporting versions and display missing metrics as NaN. Use `list_fits(FIT_ROOT)` separately when you intentionally want to browse previous experiments. Only explicitly selected IDs are propagated.

In [6]:
if saved_df.empty:
    print("No saved fits in this sweep. Inspect the raw-screening table.")
else:
    columns = ["fit_id", "fourier_box_R", "fourier_box_r", "n_modes",
               "raw_rmse", "v_rmse", "force_combined_rmse", "path"]
    display(saved_df[columns].sort_values("v_rmse"))

,fit_id,fourier_box_R,fourier_box_r,n_modes,raw_rmse,v_rmse,force_combined_rmse,path
3,fit_342a4798415698a4,40.0,80.0,50,0.000187,0.001286,0.002458,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
1,fit_d250fb158c7e93db,20.0,80.0,50,0.000191,0.001497,0.018140,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
5,fit_f33d5a618d552ac9,60.0,80.0,50,0.000188,0.001659,0.001509,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
7,fit_058cc3aeeede3a81,80.0,80.0,50,0.000189,0.002595,0.002123,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
4,fit_701eb8a6de657f7c,60.0,60.0,50,0.000494,0.008097,0.003602,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
6,fit_7277d62b90160302,80.0,60.0,50,0.000494,0.008097,0.003550,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
2,fit_9600c945824059c7,40.0,60.0,50,0.000492,0.010628,0.005226,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...
0,fit_eda7b7b165d14318,20.0,60.0,50,0.000492,0.010803,0.026274,/home/iren/quantum_algorithm/Shin-Metiu/bosoni...


In [9]:
selected = settings.get("selected_for_dynamics", [])
if not selected:
    print("No classical dynamics requested. Set selected_for_dynamics explicitly.")
else:
    validated = selected_fit_classical_validation(selected, fit_root=FIT_ROOT, result_root=RESULT_ROOT)
    for run in validated:
        print(run["fit_id"], "exact cached:", run["exact"][2], "fitted cached:", run["fitted"][2])

fit_f33d5a618d552ac9 exact cached: True fitted cached: True
fit_342a4798415698a4 exact cached: True fitted cached: False
